In [1]:
from A3_p2_functions import *
from harris import *

# Part A

In [2]:
re_define_points = False

In [3]:
img_path_1 = 'part_1_images/img_left.jpg'
img_path_2 = 'part_1_images/img_right.jpg'

In [4]:
if re_define_points:
    chosen_points_1 = label_image(img_path_1)
    chosen_points_2 = label_image(img_path_2)

In [5]:
left_points_path = 'part_1_results/img_left_points.json'
right_points_path = 'part_1_results/img_right_points.json'

In [6]:
with open(left_points_path, 'r') as read_file:
    img_1_pts = np.array(json.load(read_file))
read_file.close()

with open(right_points_path, 'r') as read_file:
    img_2_pts = np.array(json.load(read_file))
read_file.close()

In [7]:
homography = compute_homography(img_1_pts, img_2_pts)

In [8]:
print("Homography: ", homography)

Homography:  [[ 1.65112933e+00 -7.56993875e-02 -4.35396789e+02]
 [ 3.28986805e-01  1.36029289e+00 -1.34210563e+02]
 [ 1.01295189e-03 -3.05509450e-04  1.00000000e+00]]


In [9]:
with open(left_points_path, 'r') as read_file:
    img_1_pts = np.array(json.load(read_file))
read_file.close()

with open(right_points_path, 'r') as read_file:
    img_2_pts = np.array(json.load(read_file))
read_file.close()

In [10]:
pts1_h = np.column_stack([img_1_pts, np.ones(len(img_1_pts))])

mapped = (homography @ pts1_h.T).T
mapped = mapped[:, :2] / mapped[:, 2:3]

# print(np.abs(mapped - img_2_pts))

In [11]:
img_1 = cv2.imread('part_1_images/img_left.jpg')
img_2 = cv2.imread('part_1_images/img_right.jpg')

method = 'nearest_neighbor'

In [12]:
w_1, w_2, mosaic = warp_2_imgs(img_1, img_2, img_1_pts, img_2_pts, method='bilinear', img_name='warped_imgs',create_mosaic=True, gain_correct=False)

# Part B

In [13]:
img_path_left = 'part_1_images/img_left.jpg'
img_path_right = 'part_1_images/img_right.jpg'

img_left = cv2.imread(img_path_left)
img_right = cv2.imread(img_path_right)

img_left_bw = cv2.cvtColor(img_left, cv2.COLOR_BGR2GRAY)
img_right_bw = cv2.cvtColor(img_right, cv2.COLOR_BGR2GRAY)

In [14]:
h_1, coords_1 = get_harris_corners(img_left_bw)
h_2, coords_2 = get_harris_corners(img_right_bw)

In [15]:
# corners_img = plot_harris_corners(img_path_left, h_1, coords_1, corners_thresh=0.01)

# corners_img = plot_harris_corners(img_path_left, h_2, coords_2, corners_thresh=0.01)

In [16]:
ANMS_coords_1 = ANMS(h_1, coords_1)
ANMS_coords_2 = ANMS(h_2, coords_2)

In [17]:
# ANMS_corners_img = plot_harris_corners(img_path_left, h_1, ANMS_coords_1, img_name='ANMS')

# ANMS_corners_img = plot_harris_corners(img_path_left, h_1, ANMS_coords_1, img_name='ANMS')

In [18]:
vecs_left, patches_left = feature_descript_extract(img_left_bw, ANMS_coords_1)

vecs_right, patches_right = feature_descript_extract(img_right_bw, ANMS_coords_2)

In [19]:
matched_pairs = match_features(vecs_left, vecs_right)

In [20]:
coords_1_ordered = ANMS_coords_1[::-1].T
coords_2_ordered = ANMS_coords_2[::-1].T

In [21]:
NUM_ITERATIONS = 5000

In [22]:
final_homography, max_inliers_mask = RANSAC4(matched_pairs, coords_1_ordered, coords_2_ordered, NUM_ITERATIONS=NUM_ITERATIONS)

RANSAC done, max_inliers found =  125


In [23]:
final_homography

array([[ 1.96466757e+00, -5.51648112e-02, -5.36011421e+02],
       [ 4.14413669e-01,  1.61824543e+00, -1.79189026e+02],
       [ 1.38407746e-03, -7.93067926e-05,  1.00000000e+00]])